# 05 — Feature EDA

Compute handcrafted features per trial and look for group differences.

**Read this first.** PD trials in this dataset are ~10% longer than control
trials. Any duration-confounded feature (energy sums, band-energy sums)
will look discriminative for the wrong reason. `src/features.py` was
rewritten to be duration-invariant; this notebook also plots the duration
distribution as the first sanity check so the confound is impossible to
miss.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio

plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (10, 4)

from tqdm.notebook import tqdm
from scipy.stats import mannwhitneyu
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from src.data_loader import iter_trials, load_trial
from src.features import per_window_features


## 0. Duration confound — verify before featurising

If the medians differ by more than a couple of seconds, every energy-summed feature is contaminated. The next sections use `src.features` which sidesteps this by using means and ratios only.

In [ ]:
trials = list(iter_trials())
dur_rows = []
for tp in tqdm(trials, desc="durations"):
    try:
        m = sio.loadmat(tp.path, variable_names=["t_axis_target"], squeeze_me=True)
        t = np.asarray(m["t_axis_target"]).ravel()
        dur_rows.append({"subject_id": tp.subject_id, "group": tp.group,
                         "duration_s": float(t.max() - t.min())})
    except Exception as e:
        print(f"  skip {tp.path}: {e}")
dur_df = pd.DataFrame(dur_rows)
print(dur_df.groupby("group")["duration_s"].describe().round(2))


## 1. Featurise every trial (whole spectrogram)

Duration-invariant features only (`src.features.per_window_features`).

In [ ]:
LIMIT = None  # set e.g. 50 for a fast smoke test
sample = trials if LIMIT is None else trials[:LIMIT]

rows = []
for tp in tqdm(sample, desc="featurising"):
    try:
        tr = load_trial(tp.path)
        feats = per_window_features(tr["ce_foot"], tr["ce_torso"], tr["doppler"])
        feats.update({
            "subject_id": tp.subject_id, "group": tp.group,
            "test": tp.test, "trial": tp.trial,
        })
        rows.append(feats)
    except Exception as e:
        print(f"  skip {tp.path}: {e}")

feat_df = pd.DataFrame(rows)
print(f"{len(feat_df)} trials × {feat_df.shape[1] - 4} features")
feat_df.head()


## 2. Per-feature group difference (Mann–Whitney U)

If the top hits now look like spectral-shape features (centroid, bandwidth, entropy, band ratios) rather than energy sums, the duration leak is gone.

In [ ]:
feature_cols = [c for c in feat_df.columns if c not in
                ("subject_id", "group", "test", "trial")]

results = []
for c in feature_cols:
    a = feat_df.loc[feat_df["group"] == "control", c].dropna()
    b = feat_df.loc[feat_df["group"] == "pd", c].dropna()
    if a.size < 5 or b.size < 5:
        continue
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    results.append({"feature": c, "p_value": p,
                    "ctrl_median": a.median(), "pd_median": b.median(),
                    "diff_pct": 100 * (b.median() - a.median()) / (abs(a.median()) + 1e-12)})
diffs = pd.DataFrame(results).sort_values("p_value")
diffs.head(15)


## 3. Distribution plots for the top features

In [ ]:
top_feats = diffs.head(6)["feature"].tolist()
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, c in zip(axes.flat, top_feats):
    for grp, colour in [("control", "#4c8"), ("pd", "#e66")]:
        vals = feat_df.loc[feat_df["group"] == grp, c].dropna()
        ax.hist(vals, bins=20, alpha=0.6, label=grp, color=colour)
    ax.set_title(c, fontsize=9)
    ax.legend(fontsize=7)
plt.tight_layout(); plt.show()


## 4. Correlation between top features and trial duration

Last safety check: if any 'discriminative' feature still correlates strongly with duration, treat its p-value with suspicion.

In [ ]:
feat_df = feat_df.merge(dur_df[["subject_id", "duration_s"]].drop_duplicates(),
                       on="subject_id", how="left")
corr_to_dur = (feat_df[feature_cols + ["duration_s"]].corr()["duration_s"]
               .drop("duration_s").sort_values(key=lambda s: s.abs(), ascending=False))
print(corr_to_dur.head(10))


## 5. Correlation heatmap (top 15 features)

In [ ]:
subset = diffs.head(15)["feature"].tolist()
corr = feat_df[subset].corr()
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr, cmap="seismic", vmin=-1, vmax=1)
ax.set_xticks(range(len(subset))); ax.set_xticklabels(subset, rotation=90, fontsize=7)
ax.set_yticks(range(len(subset))); ax.set_yticklabels(subset, fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.04)
ax.set_title("Feature correlation")
plt.tight_layout(); plt.show()


## 6. PCA projection coloured by group

In [ ]:
X = feat_df[feature_cols].values
y = (feat_df["group"] == "pd").astype(int).values
mask = ~np.isnan(X).any(axis=1)
X, y = X[mask], y[mask]

X_s = StandardScaler().fit_transform(X)
pca = PCA(n_components=2)
X_p = pca.fit_transform(X_s)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X_p[y == 0, 0], X_p[y == 0, 1], c="#4c8", label="control", s=25, alpha=0.7)
ax.scatter(X_p[y == 1, 0], X_p[y == 1, 1], c="#e66", label="PD", s=25, alpha=0.7)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.legend()
ax.set_title("PCA of duration-invariant features")
plt.tight_layout(); plt.show()


## 7. Save the feature table

In [ ]:
out = ROOT / "outputs" / "metrics" / "trial_features.csv"
out.parent.mkdir(parents=True, exist_ok=True)
feat_df.to_csv(out, index=False)
print(f"saved {out}")


### Notes

- p-values are uncorrected; this is exploratory, not confirmatory. Don't claim significance from the raw values.
- Trial features are not subject features — multiple trials per subject inflate degrees of freedom. The SVM/RF baselines in Notebook 07 mitigate this with subject-level LOSO-CV.
- If `corr_to_dur` (section 4) shows any |r| > 0.3 for a "discriminative" feature, that feature is still tangled with duration. The current feature set should keep all those correlations low.
